# Fix under-segmented cells (3D Cellpose + membrane) → Ultrack tracking

Run it top to bottom. You only need to change the **Settings** cell.

1. **Cleanup** of z-stitching artifacts: labels broken into pieces, tiny fragments, and flat 1–2-slice slabs.
2. **Split** each cell along the membrane. The cut is placed in the middle of the membrane (`CYTO_THR`). Pieces are merged back unless a real membrane wall separates them
   (wall score ≥ `WALL_THR`) and each piece is big enough (≥ `MIN_VOL`) and thick enough (≥ `MIN_THICK`).
3. **Test on one frame** and look at the splits, then **run all frames** in parallel. Re-running skips frames that are already done.
4. **QA** across the movie, then a **review gallery** of the least confident splits.
5. **Ultrack**: tracks using *both* the fixed and the original labels as candidate segmentations, plus the membrane signal.

Input, either:
* **zarr** (default): one container with the membrane images for all timepoints and one with the labels
  (e.g. `T, Z, Y, X` or OME-Zarr `T, C, Z, Y, X`), or
* **tif**: `<DATA_DIR>/tNNN_image.tif` + `<DATA_DIR>/tNNN_labels.tif` (one ZYX file per timepoint).

Outputs are written per timepoint (`OUT_DIR/fixed/tNNN_labels_fixed.tif`) and also collected into
`OUT_DIR/fixed_labels.zarr` (T, Z, Y, X) at the end of section 2.

Packages: `numpy scipy scikit-image pandas tifffile matplotlib joblib` (plus `ultrack zarr` for section 5).

In [ ]:
# ---------------- Settings ----------------
INPUT_FORMAT = "zarr"          # "zarr" or "tif"

# zarr input
IMAGE_ZARR = "data/images.zarr"   # membrane images, all timepoints
LABEL_ZARR = "data/labels.zarr"   # Cellpose labels, all timepoints
IMAGE_KEY  = None   # array path inside the container, e.g. "0" (OME-Zarr full resolution) or "membrane"; None = auto
LABEL_KEY  = None
IMAGE_AXES = None   # e.g. "TZYX" or "TCZYX"; None = auto (4D -> TZYX, 5D -> TCZYX)
LABEL_AXES = None
CHANNEL    = 0      # which channel is the membrane, if the image has a C axis

# tif input (one file per timepoint)
DATA_DIR = "data"
IMAGE_SUFFIX = "_image.tif"
LABEL_SUFFIX = "_labels.tif"

OUT_DIR  = "out/notebook"     # fixed labels, flags, QA, ultrack output go here

VOXEL = (0.4374, 0.2158, 0.2158)   # z, y, x in µm

# splitting parameters
WALL_THR   = 1.0    # membrane wall score needed to keep a split (lower = more splits)
MIN_VOL    = 60.0   # µm³, smallest piece a split may create (median cell here ≈ 145 µm³)
MIN_THICK  = 1.0    # µm, every piece must be at least this thick (stops thin rim shells being cut off)
H_FRAC     = 0.12   # seed sensitivity (h-minima depth, relative to the cell's membrane contrast)
SMOOTH_UM  = 0.35   # µm, membrane smoothing before the watershed
CYTO_THR   = 0.5    # 0 = cytoplasm, 1 = membrane: below this counts as cytoplasm when placing the cut
                    # (the cut goes through the middle of the membrane, furthest from cytoplasm on both sides)

# cleanup parameters
SLIVER_VOL = 25.0   # µm³, smaller objects are treated as fragments
SLAB_THICK = 0.6    # µm, thinner objects (1–2 z-slices) are z-stitching slabs

N_JOBS = 4          # frames processed in parallel (~1.5 GB RAM each)

In [ ]:
import os, glob, time
import numpy as np, pandas as pd, tifffile
import matplotlib.pyplot as plt
from scipy import ndimage as ndi
from skimage.segmentation import watershed, find_boundaries
from skimage.morphology import h_minima
from joblib import Parallel, delayed

SP = np.array(VOXEL, float)
VV = float(np.prod(SP))
FIX_DIR = f"{OUT_DIR}/fixed"
os.makedirs(FIX_DIR, exist_ok=True)

def _open_zarr(path, key):
    import zarr
    z = zarr.open(path, mode="r")
    if hasattr(z, "shape"): return z, path                       # the container is itself an array
    if key is not None: return z[key], f"{path}/{key}"
    arrays = {k: v for k, v in z.arrays()} if hasattr(z, "arrays") else {}
    if "0" in arrays: return arrays["0"], f"{path}/0"            # OME-Zarr: full resolution
    if len(arrays) == 1: k = next(iter(arrays)); return arrays[k], f"{path}/{k}"
    groups = [k for k, _ in z.groups()] if hasattr(z, "groups") else []
    if len(arrays) == 0 and len(groups) == 1:                     # e.g. OME-Zarr nested one level down
        return _open_zarr(f"{path}/{groups[0]}", None)
    raise ValueError(f"{path}: can't pick an array automatically. Arrays: {list(arrays)}, groups: {groups}. "
                     f"Set IMAGE_KEY / LABEL_KEY.")

def _axes(arr, axes):
    if axes is None:
        axes = {4: "TZYX", 5: "TCZYX"}.get(arr.ndim)
        if axes is None: raise ValueError(f"array has {arr.ndim} dims {arr.shape}; set IMAGE_AXES / LABEL_AXES")
    axes = axes.upper()
    assert len(axes) == arr.ndim and set("TZYX") <= set(axes), f"axes {axes} don't fit shape {arr.shape}"
    return axes

def _frame(arr, axes, ti, channel=0):
    idx = tuple(ti if a == "T" else channel if a == "C" else slice(None) for a in axes)
    rest = [a for a in axes if a not in "TC"]
    return np.transpose(np.asarray(arr[idx]), [rest.index(a) for a in "ZYX"])

if INPUT_FORMAT == "zarr":
    _img_arr, _img_src = _open_zarr(IMAGE_ZARR, IMAGE_KEY); _img_axes = _axes(_img_arr, IMAGE_AXES)
    _lab_arr, _lab_src = _open_zarr(LABEL_ZARR, LABEL_KEY); _lab_axes = _axes(_lab_arr, LABEL_AXES)
    n_t = _img_arr.shape[_img_axes.index("T")]
    assert n_t == _lab_arr.shape[_lab_axes.index("T")], "images and labels have a different number of timepoints"
    FRAMES = [f"t{i:03d}" for i in range(n_t)]
    _TI = {t: i for i, t in enumerate(FRAMES)}
    load_image  = lambda t: _frame(_img_arr, _img_axes, _TI[t], CHANNEL)
    load_labels = lambda t: _frame(_lab_arr, _lab_axes, _TI[t])
    print(f"images: {_img_src}  shape {_img_arr.shape}  axes {_img_axes}  dtype {_img_arr.dtype}"
          + (f"  -> channel {CHANNEL}" if "C" in _img_axes else ""))
    print(f"labels: {_lab_src}  shape {_lab_arr.shape}  axes {_lab_axes}  dtype {_lab_arr.dtype}")
    _zyx_i = load_image(FRAMES[0]).shape; _zyx_l = load_labels(FRAMES[0]).shape
    assert _zyx_i == _zyx_l, f"image frame {_zyx_i} and label frame {_zyx_l} differ - check the axes settings"
    print(f"one timepoint = Z, Y, X {_zyx_i}")
else:
    FRAMES = sorted(os.path.basename(p)[: -len(IMAGE_SUFFIX)] for p in glob.glob(f"{DATA_DIR}/*{IMAGE_SUFFIX}"))
    load_image  = lambda t: tifffile.imread(f"{DATA_DIR}/{t}{IMAGE_SUFFIX}")
    load_labels = lambda t: tifffile.imread(f"{DATA_DIR}/{t}{LABEL_SUFFIX}")
    missing = [t for t in FRAMES if not os.path.exists(f"{DATA_DIR}/{t}{LABEL_SUFFIX}")]
    if missing: print("WARNING, no labels for:", missing)
print(f"{len(FRAMES)} timepoints: {FRAMES[:3]} ... {FRAMES[-3:]}")

## Functions
`split_cell` works on one cell crop. `cleanup` and `process_frame` work on a whole timepoint.

In [ ]:
def _interfaces(parts):
    # {(a, b): [(axis, index), ...]} voxel faces between parts a and b (6-connectivity)
    out = {}
    for ax in range(3):
        a = np.moveaxis(parts, ax, 0); A, B = a[:-1], a[1:]
        m = (A != B) & (A > 0) & (B > 0)
        if not m.any(): continue
        for p, q, i in zip(A[m], B[m], np.argwhere(m)):
            out.setdefault((min(p, q), max(p, q)), []).append((ax, i))
    return out

def _wall_scores(parts, gn):
    # mean normalised membrane intensity on each interface (max of the two facing voxels)
    res = {}
    for k, lst in _interfaces(parts).items():
        vals = []
        for ax, i in lst:
            a = np.moveaxis(gn, ax, 0); i2 = i.copy(); i2[0] += 1
            vals.append(max(a[tuple(i)], a[tuple(i2)]))
        res[k] = (float(np.mean(vals)), len(lst))
    return res

def split_cell(mask, img):
    # mask: bool crop of one cell; img: membrane crop. Returns (parts 1..n, [(a, b, wall_score, n_faces)])
    g = ndi.gaussian_filter(img.astype(np.float32), SMOOTH_UM / SP)
    dt = ndi.distance_transform_edt(mask, sampling=SP)
    rim = np.median(g[mask & (dt <= 0.5)])
    core = np.percentile(g[mask], 20)
    gn = (g - core) / max(rim - core, 1e-6)          # 0 = cytoplasm, 1 = this cell's own rim
    seeds = ndi.label(h_minima(np.where(mask, gn, gn.max()), H_FRAC) & mask)[0]
    if seeds.max() < 2:
        return mask.astype(np.int32), []
    # Cut through the MIDDLE of the membrane. A tilted membrane looks like a bright band several
    # z-slices thick; flooding the raw intensity lets one piece spill through that band into the other
    # (e.g. t001 labels 112/113). Flooding the distance from cytoplasm puts the cut at the band's centre.
    dark = mask & (gn < CYTO_THR)
    D = ndi.distance_transform_edt(~dark, sampling=SP)   # µm from nearest cytoplasm
    parts = watershed(D + 0.01 * gn, seeds, mask=mask)  # gn only breaks ties inside cytoplasm
    while True:                                      # merge back weak / small / thin pieces
        labs = [l for l in np.unique(parts) if l > 0]
        if len(labs) < 2: break
        vols, thick = {}, {}
        for l, o in zip(range(1, parts.max() + 1), ndi.find_objects(parts)):
            if o is None: continue
            pp = np.pad(parts[o] == l, 1)
            vols[l] = pp.sum() * VV
            thick[l] = ndi.distance_transform_edt(pp, sampling=SP).max()
        score = _wall_scores(parts, gn)
        if not score: break
        bad = [l for l in labs if vols[l] < MIN_VOL or thick[l] < MIN_THICK]
        if bad:
            l = min(bad, key=lambda q: (thick[q] >= MIN_THICK, vols[q]))
            nb = [(k, s) for k, s in score.items() if l in k]
            if not nb: parts[parts == l] = 0; continue
            k = max(nb, key=lambda x: x[1][1])[0]     # neighbour with largest contact
        else:
            k, (s, _) = min(score.items(), key=lambda x: x[1][0])
            if s >= WALL_THR: break
        parts[parts == k[1]] = k[0]
    u = np.unique(parts); u = u[u > 0]
    lut = np.zeros(parts.max() + 1, np.int32); lut[u] = np.arange(1, len(u) + 1); parts = lut[parts]
    if (mask & (parts == 0)).any() and parts.max() > 0:   # refill voxels of dropped pieces
        _, ind = ndi.distance_transform_edt(parts == 0, sampling=SP, return_indices=True)
        parts = np.where(mask, parts[tuple(ind)], 0)
    walls = [(a, b, s, n) for (a, b), (s, n) in _wall_scores(parts, gn).items()]
    return parts, walls

In [ ]:
def _contact_neighbour(lab, region, exclude):
    ring = ndi.binary_dilation(region) & ~region
    v = lab[ring]; v = v[(v > 0) & (v != exclude)]
    if v.size == 0: return 0
    u, c = np.unique(v, return_counts=True); return int(u[c.argmax()])

def cleanup(lab):
    lab = lab.copy(); n_fix = 0
    # 1) labels in several disconnected pieces: keep the largest, give strays to the touching neighbour
    for L, s in enumerate(ndi.find_objects(lab), 1):
        if s is None: continue
        sl = tuple(slice(max(a.start - 1, 0), a.stop + 1) for a in s)
        sub = lab[sl]; cc, n = ndi.label(sub == L)
        if n > 1:
            keep = np.argmax(ndi.sum(cc > 0, cc, range(1, n + 1))) + 1
            for k in range(1, n + 1):
                if k != keep:
                    piece = cc == k; sub[piece] = _contact_neighbour(sub, piece, L); n_fix += 1
    # 2) tiny objects and flat slabs: smallest first, merge into the neighbour they touch most
    while True:
        cand = []
        for L, s in enumerate(ndi.find_objects(lab), 1):
            if s is None: continue
            m = np.pad(lab[s] == L, 1); vol = m.sum() * VV
            if vol < SLIVER_VOL or ndi.distance_transform_edt(m, sampling=SP).max() < SLAB_THICK:
                cand.append((vol, L, s))
        if not cand: break
        for vol, L, s in sorted(cand):
            sl = tuple(slice(max(a.start - 1, 0), a.stop + 1) for a in s)
            sub = lab[sl]; m = sub == L
            if m.any(): sub[m] = _contact_neighbour(sub, m, L); n_fix += 1
    return lab, n_fix

def _vol_stats(lab):
    v = np.bincount(lab.ravel())[1:] * VV; v = v[v > 0]
    return dict(n=int((v >= SLIVER_VOL).sum()), n_over300=int((v > 300).sum()),
                n_frag=int((v < SLIVER_VOL).sum()), vol_median=float(np.median(v[v >= SLIVER_VOL])))

def process_frame(t, overwrite=False):
    f_lab, f_csv = f"{FIX_DIR}/{t}_labels_fixed.tif", f"{FIX_DIR}/{t}_flags.csv"
    if os.path.exists(f_csv) and not overwrite:
        return t, "skipped"
    t0 = time.time()
    img = load_image(t).astype(np.float32)
    lab0 = load_labels(t).astype(np.int32)
    lab, n_fix = cleanup(lab0)
    out = np.zeros_like(lab); nxt = 1; rows = []
    for L, s in enumerate(ndi.find_objects(lab), 1):
        if s is None: continue
        sl = tuple(slice(max(a.start - 2, 0), a.stop + 2) for a in s)
        m = lab[sl] == L; vol = m.sum() * VV
        parts, walls = split_cell(m, img[sl]) if vol >= 2 * MIN_VOL else (m.astype(np.int32), [])
        o = out[sl]
        for k in range(1, parts.max() + 1):
            o[parts == k] = nxt
            rows.append(dict(t=t, new_label=nxt, orig_label=L, n_parts=int(parts.max()),
                             vol_um3=round(float((parts == k).sum() * VV), 1), orig_vol_um3=round(float(vol), 1),
                             wall_scores=";".join(f"{w[2]:.2f}" for w in walls),
                             min_wall=round(min(w[2] for w in walls), 2) if walls else np.nan))
            nxt += 1
    tifffile.imwrite(f_lab, out.astype(np.uint32), compression="zlib")
    df = pd.DataFrame(rows)
    before, after = _vol_stats(lab0), _vol_stats(out)
    summ = dict(t=t, n_before=before["n"], n_after=after["n"],
                n_split=int(df.drop_duplicates("orig_label").n_parts.gt(1).sum()), n_fragment_fixes=n_fix,
                over300_before=before["n_over300"], over300_after=after["n_over300"],
                frag_before=before["n_frag"], frag_after=after["n_frag"],
                vol_median_after=after["vol_median"], seconds=round(time.time() - t0, 1))
    pd.DataFrame([summ]).to_csv(f"{FIX_DIR}/{t}_summary.csv", index=False)
    df.to_csv(f_csv, index=False)        # written last = "frame done" marker
    return t, f"{summ['n_before']} -> {summ['n_after']} cells, {summ['n_split']} split, {summ['seconds']} s"

In [ ]:
# ---- plotting helpers ----
def _norm(img, lab):
    lo, hi = np.percentile(img[lab > 0], [1, 99.5]); return lambda a: np.clip((a - lo) / (hi - lo), 0, 1)

COLS = np.array([[1, .2, .2], [.2, 1, .2], [1, 1, .2], [1, .3, 1], [.2, 1, 1]])

def show_splits(t, orig_labels, flags=None):
    # XY / XZ / YZ through each split cell: coloured outlines = new pieces, blue = other cells
    flags = pd.read_csv(f"{FIX_DIR}/{t}_flags.csv") if flags is None else flags
    img = load_image(t).astype(np.float32)
    fx = tifffile.imread(f"{FIX_DIR}/{t}_labels_fixed.tif")
    nm = _norm(img, fx)
    fig, ax = plt.subplots(len(orig_labels), 3, figsize=(13, 3.8 * len(orig_labels)), squeeze=False)
    for r, L in enumerate(orig_labels):
        rows = flags[flags.orig_label == L]; new = rows.new_label.values
        cell = np.isin(fx, new); idx = np.argwhere(cell)
        (z0, y0, x0), (z1, y1, x1) = idx.min(0), idx.max(0) + 1
        if len(new) > 1:   # slice through the interface between pieces
            P = np.where(cell, fx, 0); iface = find_boundaries(P, mode="inner") & cell & ~find_boundaries(cell, mode="inner")
            zc, yc, xc = np.argwhere(iface).mean(0).astype(int) if iface.any() else idx.mean(0).astype(int)
        else:
            zc, yc, xc = idx.mean(0).astype(int)
        pad = 20; Y0, Y1, X0, X1 = max(y0 - pad, 0), y1 + pad, max(x0 - pad, 0), x1 + pad; Z0, Z1 = max(z0 - 6, 0), z1 + 6
        def panel(a, im2, lab2, title, asp=1):
            rgb = np.dstack([nm(im2)] * 3)
            rgb[find_boundaries(lab2, mode="inner") & (lab2 > 0) & ~np.isin(lab2, new)] = [.3, .5, 1]
            for k, nl in enumerate(new): rgb[find_boundaries(lab2 == nl, mode="inner")] = COLS[k % 5]
            a.imshow(rgb, aspect=asp); a.set_title(title, fontsize=9); a.axis("off")
        title = f"{t}  label {L}: walls [{rows.wall_scores.iloc[0]}]  vols {list(rows.vol_um3)}"
        panel(ax[r, 0], img[zc, Y0:Y1, X0:X1], fx[zc, Y0:Y1, X0:X1], title)
        panel(ax[r, 1], img[Z0:Z1, yc, X0:X1], fx[Z0:Z1, yc, X0:X1], "XZ", SP[0] / SP[1])
        panel(ax[r, 2], img[Z0:Z1, Y0:Y1, xc], fx[Z0:Z1, Y0:Y1, xc], "YZ", SP[0] / SP[1])
    plt.tight_layout(); plt.show()

def show_before_after(t, y_fracs=(0.35, 0.5, 0.65)):
    # XZ slices, original vs fixed (random colours per label)
    img = load_image(t).astype(np.float32)
    a = load_labels(t); b = tifffile.imread(f"{FIX_DIR}/{t}_labels_fixed.tif")
    nm = _norm(img, a); rng = np.random.default_rng(1)
    zs, ys, xs = (np.where(a.any(axis=tuple(j for j in range(3) if j != d)))[0] for d in range(3))
    Z0, Z1, X0, X1 = max(zs.min() - 3, 0), zs.max() + 3, max(xs.min() - 5, 0), xs.max() + 5
    fig, ax = plt.subplots(len(y_fracs), 2, figsize=(16, 3.6 * len(y_fracs)), squeeze=False)
    for r, yf in enumerate(y_fracs):
        y = int(ys.min() + yf * (ys.max() - ys.min())); g = nm(img[Z0:Z1, y, X0:X1])[..., None]
        for c, (lab, name) in enumerate([(a, "original"), (b, "fixed")]):
            L = lab[Z0:Z1, y, X0:X1]; col = rng.random((lab.max() + 1, 3)) * .7 + .3; col[0] = 0
            rgb = np.where(L[..., None] > 0, .55 * g + .45 * col[L], g)
            rgb[find_boundaries(L, mode="inner") & (L > 0)] = 1
            ax[r, c].imshow(rgb, aspect=SP[0] / SP[1]); ax[r, c].set_title(f"{name}, XZ at y={y}"); ax[r, c].axis("off")
    plt.tight_layout(); plt.show()

## 1. Test on one frame
Check the results here before running the whole movie. If the splits look wrong, change the Settings cell
(mainly `WALL_THR`), then re-run this cell (it uses `overwrite=True`).

In [ ]:
TEST_FRAME = FRAMES[0]
print(process_frame(TEST_FRAME, overwrite=True))
display(pd.read_csv(f"{FIX_DIR}/{TEST_FRAME}_summary.csv"))
show_before_after(TEST_FRAME)

In [ ]:
# the split cells with the strongest walls (should clearly be real)...
fl = pd.read_csv(f"{FIX_DIR}/{TEST_FRAME}_flags.csv")
split = fl[fl.n_parts > 1].drop_duplicates("orig_label").sort_values("min_wall")
print(f"{len(split)} cells split in {TEST_FRAME}")
show_splits(TEST_FRAME, split.orig_label.tail(4).tolist(), fl)

In [ ]:
# ...and the weakest ones that were still accepted: if these look wrong, raise WALL_THR
show_splits(TEST_FRAME, split.orig_label.head(4).tolist(), fl)

## 2. Run all frames
Frames run in parallel (`N_JOBS`). A frame counts as done once its `_flags.csv` exists, so if the run is
interrupted, just run this cell again. **If you change parameters, delete `OUT_DIR/fixed` first**, or the
frames already done will keep their old results.

In [ ]:
t0 = time.time()
res = Parallel(n_jobs=N_JOBS, verbose=5)(delayed(process_frame)(t) for t in FRAMES)
for t, msg in res: print(t, msg)
print(f"done in {(time.time() - t0) / 60:.1f} min")

In [ ]:
# collect the fixed labels into one zarr (T, Z, Y, X), handy for napari or other tools
import zarr
shape0 = tifffile.imread(f"{FIX_DIR}/{FRAMES[0]}_labels_fixed.tif").shape
fz = zarr.open(f"{OUT_DIR}/fixed_labels.zarr", mode="w", shape=(len(FRAMES),) + shape0,
               chunks=(1,) + shape0, dtype=np.uint32)
for i, t in enumerate(FRAMES):
    fz[i] = tifffile.imread(f"{FIX_DIR}/{t}_labels_fixed.tif")
print("written", f"{OUT_DIR}/fixed_labels.zarr", fz.shape)

## 3. QA across the movie

In [ ]:
summ = pd.concat([pd.read_csv(f) for f in sorted(glob.glob(f"{FIX_DIR}/*_summary.csv"))]).reset_index(drop=True)
flags = pd.concat([pd.read_csv(f) for f in sorted(glob.glob(f"{FIX_DIR}/*_flags.csv"))])
flags.to_csv(f"{OUT_DIR}/all_flags.csv", index=False); summ.to_csv(f"{OUT_DIR}/all_summary.csv", index=False)

x = np.arange(len(summ))
step = max(1, len(x) // 10)                    # at most ~10 tick labels on long movies
annotate = len(x) <= 15                        # print values on points for short runs only
fig, ax = plt.subplots(1, 3, figsize=(17, 4.2))

def _line(a, y, label):
    a.plot(x, y, "o-", label=label)
    if annotate:
        for xi, yi in zip(x, y): a.annotate(str(int(yi)), (xi, yi), textcoords="offset points", xytext=(0, 6), ha="center", fontsize=8)

_line(ax[0], summ.n_before, "original"); _line(ax[0], summ.n_after, "fixed")
ax[0].set_title("Cells per timepoint"); ax[0].set_ylabel("number of cells (≥ 25 µm³)"); ax[0].legend(loc="lower right")

ax[1].bar(x, summ.n_split, color="tab:green")
if annotate:
    for xi, yi in zip(x, summ.n_split): ax[1].text(xi, yi, str(int(yi)), ha="center", va="bottom", fontsize=8)
ax[1].set_title("Cells split per timepoint"); ax[1].set_ylabel("number of original cells split")

_line(ax[2], summ.over300_before, "original"); _line(ax[2], summ.over300_after, "fixed")
ax[2].set_title("Objects > 300 µm³ (likely merges)"); ax[2].set_ylabel("number of objects > 300 µm³"); ax[2].legend()

for a in ax:
    a.set_ylim(0, a.get_ylim()[1] * 1.15)     # start at 0, headroom for the value labels
    a.set_xlabel("timepoint")
    a.set_xticks(x[::step]); a.set_xticklabels(summ.t[::step], rotation=45)
    a.grid(axis="y", alpha=0.3)
    a.yaxis.set_major_locator(plt.MaxNLocator(integer=True))   # counts -> whole numbers
plt.tight_layout(); plt.show()
display(summ[["n_before", "n_after", "n_split", "over300_before", "over300_after", "frag_before", "frag_after"]].sum().to_frame("total").T)
# a frame that jumps away from its neighbours in the plots above is worth a look with show_before_after(t)

In [ ]:
# volume distribution before vs after (sampled frames to stay fast on long movies)
sample = FRAMES[:: max(1, len(FRAMES) // 10)]
vb, va = [], []
for t in sample:
    for lab, acc in [(load_labels(t), vb), (tifffile.imread(f"{FIX_DIR}/{t}_labels_fixed.tif"), va)]:
        v = np.bincount(lab.ravel())[1:] * VV; acc += list(v[v > 0])
bins = np.linspace(0, 600, 61)
plt.figure(figsize=(8, 4))
plt.hist(vb, bins, alpha=.5, label=f"original (CV {np.std(vb) / np.mean(vb):.2f})")
plt.hist(va, bins, alpha=.5, label=f"fixed (CV {np.std(va) / np.mean(va):.2f})")
plt.xlabel("cell volume (µm³)"); plt.ylabel("number of cells"); plt.legend(); plt.title(f"Cell volumes ({len(sample)} sampled timepoints)"); plt.tight_layout(); plt.show()

## 4. Review low-confidence splits
Splits with a minimum wall score close to `WALL_THR` are the ones most likely to be wrong.

In [ ]:
low = flags[(flags.n_parts > 1)].drop_duplicates(["t", "orig_label"]).sort_values("min_wall")
print(f"{len(low)} split cells in total; {int((low.min_wall < WALL_THR + 0.2).sum())} with min wall < {WALL_THR + 0.2:.1f}")
display(low.head(10)[["t", "orig_label", "n_parts", "orig_vol_um3", "wall_scores"]])
for t, g in low.head(6).groupby("t"):
    show_splits(t, g.orig_label.tolist())

In [ ]:
# OPTIONAL: inspect in napari (pip install napari[all]); not needed for the pipeline
# import napari
# t = FRAMES[0]
# v = napari.Viewer()
# v.add_image(load_image(t), scale=VOXEL, name="membrane")
# v.add_labels(load_labels(t), scale=VOXEL, name="original", visible=False)
# v.add_labels(tifffile.imread(f"{FIX_DIR}/{t}_labels_fixed.tif"), scale=VOXEL, name="fixed")

## 5. Track with Ultrack
For each frame, the inputs are the boundaries from **both** label sets (where both agree: 1.0; a new split only: 0.5)
blended with the membrane signal. Ultrack then picks the segmentation that is most consistent over time.
Everything is cropped to the tissue and written to `inputs.zarr` one frame at a time, so a long movie never
has to fit in RAM. The tracking is solved in overlapping windows of `WINDOW` frames.

On macOS/Windows, if the cell fails with a multiprocessing error, set `ULTRACK_WORKERS = 1`.

In [ ]:
import zarr
from ultrack import MainConfig, Tracker
from ultrack.utils import labels_to_contours

UT_DIR = f"{OUT_DIR}/ultrack"
ULTRACK_WORKERS = 2
WINDOW = 20                # frames per tracking window (ignored if the movie is shorter)
MAX_MOVE_UM = 8.0          # max distance a cell can move between frames (µm)
DIVISION_WEIGHT = -0.001   # more negative (e.g. -0.1 ... -1) = fewer divisions
os.makedirs(UT_DIR, exist_ok=True)

# crop box covering the tissue in every frame
bb = None
for t in FRAMES:
    f = tifffile.imread(f"{FIX_DIR}/{t}_labels_fixed.tif") > 0
    cur = np.array([[i.min(), i.max() + 1] for i in (np.where(f.any(axis=tuple(j for j in range(3) if j != d)))[0] for d in range(3))])
    bb = cur if bb is None else np.c_[np.minimum(bb[:, 0], cur[:, 0]), np.maximum(bb[:, 1], cur[:, 1])]
FULL_SHAPE = f.shape
CROP = tuple(slice(max(lo - m, 0), min(hi + m, n)) for (lo, hi), m, n in zip(bb, (4, 15, 15), FULL_SHAPE))
print("crop (z, y, x):", [(s.start, s.stop) for s in CROP])

shape = (len(FRAMES),) + tuple(s.stop - s.start for s in CROP)
root = zarr.open_group(f"{UT_DIR}/inputs.zarr", mode="a")
if "foreground" not in root or root["foreground"].shape != shape:
    root.create_array("foreground", shape=shape, chunks=(1,) + shape[1:], dtype=bool, overwrite=True)
    root.create_array("contours", shape=shape, chunks=(1,) + shape[1:], dtype=np.float32, overwrite=True)
    root.attrs["done"] = []
foreground, contours = root["foreground"], root["contours"]
done = set(root.attrs.get("done", []))
for i, t in enumerate(FRAMES):
    if t in done: continue
    fx = tifffile.imread(f"{FIX_DIR}/{t}_labels_fixed.tif")[CROP].astype(np.int32)
    og = load_labels(t)[CROP].astype(np.int32)
    fg, ct = labels_to_contours([fx[None], og[None]], sigma=(1.0, 2.0, 2.0))
    g = ndi.gaussian_filter(load_image(t)[CROP].astype(np.float32), (0.8, 1.6, 1.6))
    lo, hi = np.percentile(g[fx > 0], [5, 99.5])
    foreground[i] = np.asarray(fg[0])
    contours[i] = 0.5 * np.asarray(ct[0]) + 0.5 * np.clip((g - lo) / (hi - lo), 0, 1)
    done.add(t); root.attrs["done"] = sorted(done)
print("inputs ready:", len(done), "frames")

In [ ]:
cfg = MainConfig()
cfg.data_config.working_dir = UT_DIR
cfg.segmentation_config.min_area = int(MIN_VOL / VV)      # areas are in voxels
cfg.segmentation_config.max_area = int(450 / VV)
cfg.segmentation_config.threshold = 0.5
cfg.segmentation_config.n_workers = ULTRACK_WORKERS
cfg.linking_config.max_distance = MAX_MOVE_UM             # µm, because scale is passed below
cfg.linking_config.max_neighbors = 5
cfg.linking_config.n_workers = ULTRACK_WORKERS
cfg.tracking_config.division_weight = DIVISION_WEIGHT
if WINDOW and WINDOW < len(FRAMES):
    cfg.tracking_config.window_size = WINDOW
    cfg.tracking_config.overlap_size = 2

t0 = time.time()
tracker = Tracker(cfg)
tracker.track(foreground=foreground, contours=contours, scale=VOXEL, overwrite=True)
print(f"tracking took {(time.time() - t0) / 60:.1f} min")

In [ ]:
tracks = tracker.to_pandas()
tracks.to_csv(f"{UT_DIR}/tracks.csv", index=False)          # z, y, x in µm, cropped frame
with open(f"{UT_DIR}/crop_offset_zyx.txt", "w") as fh: fh.write(",".join(str(s.start) for s in CROP))
segm = tracker.to_zarr(tracks_df=tracks, store_or_path=f"{UT_DIR}/tracked.zarr", overwrite=True)
for i, t in enumerate(FRAMES):     # tracked labels back in the original image frame, label = track_id
    full = np.zeros(FULL_SHAPE, np.uint32); full[CROP] = np.asarray(segm[i])
    tifffile.imwrite(f"{UT_DIR}/{t}_tracked.tif", full, compression="zlib")

L = tracks.groupby("track_id").size()
print(f"{L.size} tracks; {int((L == len(FRAMES)).sum())} span all {len(FRAMES)} frames; "
      f"{int((tracks.drop_duplicates('track_id').parent_track_id > 0).sum())} division events")
fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
ax[0].plot(FRAMES, tracks.groupby("t").size().values, "o-"); ax[0].set_title("Tracked cells per timepoint")
ax[0].set_xlabel("timepoint"); ax[0].set_ylabel("number of tracked cells"); ax[0].set_ylim(bottom=0)
ax[0].set_xticks(FRAMES[:: max(1, len(FRAMES) // 10)]); ax[0].tick_params(axis="x", rotation=45)
ax[1].hist(L, bins=np.arange(1, len(FRAMES) + 2) - 0.5); ax[1].set_title("Track length")
ax[1].set_xlabel("track length (timepoints)"); ax[1].set_ylabel("number of tracks")
plt.tight_layout(); plt.show()